# Deep dive 4/13: `service_agents`

Source of truth for what Silver *should* do: `data_pipelines/silver/table_specs.py`'s `service_agents_spec`. Source of truth for what the data *actually* looks like: this notebook, run against your real `data/latam_bank.duckdb`. Data dictionary declares 1,200 rows, Source: Internal, Partition: monthly_snapshot.

Two things make this table worth extra attention: `assigned_branch_id` is the *other* column from the branch_id FK investigation (`customers.registration_branch_id` was the first) -- section 7 reproduces that finding directly here. And `contracts.py` currently declares **zero domains** for this table (`native_accent`, `agent_type`, `experience_level`, `agent_status`, `work_shift` all have dictionary-declared value sets but none are gated) -- section 5 checks all five against the dictionary so you can decide whether any are worth adding, the same way `customer_status`/`product_status` already are.

Run top to bottom. Checklist near the end; free cells after that.

In [1]:
import duckdb
import pandas as pd

pd.set_option("display.max_rows", 1000)
pd.set_option("display.min_rows", 1000)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", 100)

con = duckdb.connect("data/latam_bank.duckdb")

def q(sql, params=None):
    """Run a query and return a pandas DataFrame."""
    return con.execute(sql, params or []).df()

def q_full(sql, params=None):
    """Like q(), but shows every cell's full text with no truncation."""
    with pd.option_context("display.max_colwidth", None):
        return q(sql, params)

## 1. Column inventory

In [2]:
expected_silver_columns = [
    "agent_id", "employee_code", "first_name", "last_name", "email", "phone", "native_accent",
    "country_of_origin", "assigned_branch_id", "agent_type", "experience_level", "languages",
    "specialty", "hire_date", "avg_csat", "total_monthly_interactions", "agent_status", "work_shift",
]

actual_silver_columns = q("""
    SELECT column_name FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_service_agents'
    ORDER BY ordinal_position
""")["column_name"].tolist()

missing = [c for c in expected_silver_columns if c not in actual_silver_columns]
extra = [c for c in actual_silver_columns if c not in expected_silver_columns]
print("Missing from silver.dim_service_agents:", missing or "none")
print("Extra in silver.dim_service_agents:", extra or "none")
print()
print("Column order as built:", actual_silver_columns)

Missing from silver.dim_service_agents: none
Extra in silver.dim_service_agents: none

Column order as built: ['agent_id', 'employee_code', 'first_name', 'last_name', 'email', 'phone', 'native_accent', 'country_of_origin', 'assigned_branch_id', 'agent_type', 'experience_level', 'languages', 'specialty', 'hire_date', 'avg_csat', 'total_monthly_interactions', 'agent_status', 'work_shift']


In [3]:
print("Bronze columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='bronze' AND table_name='service_agents' ORDER BY ordinal_position
"""))

print("Silver columns:")
display(q("""
    SELECT column_name, data_type FROM information_schema.columns
    WHERE table_schema='silver' AND table_name='dim_service_agents' ORDER BY ordinal_position
"""))

Bronze columns:


,column_name,data_type
0,agent_id,VARCHAR
1,employee_code,VARCHAR
2,first_name,VARCHAR
3,last_name,VARCHAR
4,email,VARCHAR
5,phone,VARCHAR
6,native_accent,VARCHAR
7,country_of_origin,VARCHAR
8,assigned_branch_id,VARCHAR
9,agent_type,VARCHAR


Silver columns:


,column_name,data_type
0,agent_id,VARCHAR
1,employee_code,VARCHAR
2,first_name,VARCHAR
3,last_name,VARCHAR
4,email,VARCHAR
5,phone,VARCHAR
6,native_accent,VARCHAR
7,country_of_origin,VARCHAR
8,assigned_branch_id,VARCHAR
9,agent_type,VARCHAR


## 2. Row-count reconciliation

In [4]:
bronze_total = q("SELECT count(*) AS n FROM bronze.service_agents")["n"][0]
bronze_distinct_pk = q("SELECT count(DISTINCT agent_id) AS n FROM bronze.service_agents")["n"][0]
silver_total = q("SELECT count(*) AS n FROM silver.dim_service_agents")["n"][0]
silver_distinct_pk = q("SELECT count(DISTINCT agent_id) AS n FROM silver.dim_service_agents")["n"][0]

print(f"bronze.service_agents:      {bronze_total:,} rows, {bronze_distinct_pk:,} distinct agent_id")
print(f"silver.dim_service_agents:  {silver_total:,} rows, {silver_distinct_pk:,} distinct agent_id")
print()
print("Dictionary declares 1,200 rows -- matches" if bronze_total == 1200 else f"Dictionary declares 1,200 rows -- MISMATCH, Bronze has {bronze_total}")
print("Duplicate raw rows in Bronze:", bronze_total - bronze_distinct_pk)
print("OK -- Silver row count matches Bronze distinct PK count" if silver_total == bronze_distinct_pk
      else "MISMATCH -- investigate")

bronze.service_agents:      1,200 rows, 1,200 distinct agent_id
silver.dim_service_agents:  1,200 rows, 1,200 distinct agent_id

Dictionary declares 1,200 rows -- matches
Duplicate raw rows in Bronze: 0
OK -- Silver row count matches Bronze distinct PK count


## 3. NOT NULL audit

Dictionary NOT NULL: `agent_id`, `employee_code`, `first_name`, `last_name`, `email`, `native_accent`, `country_of_origin`, `agent_type`, `experience_level`, `languages`, `hire_date`, `agent_status`, `work_shift`. Nullable: `phone`, `assigned_branch_id` (notably -- unlike `customers.registration_branch_id`, the dictionary does NOT mark this one NOT NULL), `specialty`, `avg_csat`, `total_monthly_interactions`.

In [5]:
declared_not_null = [
    "agent_id", "employee_code", "first_name", "last_name", "email", "native_accent",
    "country_of_origin", "agent_type", "experience_level", "languages", "hire_date",
    "agent_status", "work_shift",
]

rows = []
total = q("SELECT count(*) AS n FROM bronze.service_agents")["n"][0]
for col in declared_not_null:
    nulls = q(f"""
        SELECT count(*) AS n FROM bronze.service_agents
        WHERE {col} IS NULL OR trim(CAST({col} AS VARCHAR)) = ''
    """)["n"][0]
    rows.append({"column": col, "declared": "NOT NULL", "actual_nulls": nulls,
                 "null_rate": round(nulls / total, 4) if total else None})

pd.DataFrame(rows)

,column,declared,actual_nulls,null_rate
0,agent_id,NOT NULL,0,0.0
1,employee_code,NOT NULL,0,0.0
2,first_name,NOT NULL,0,0.0
3,last_name,NOT NULL,0,0.0
4,email,NOT NULL,0,0.0
5,native_accent,NOT NULL,0,0.0
6,country_of_origin,NOT NULL,0,0.0
7,agent_type,NOT NULL,0,0.0
8,experience_level,NOT NULL,0,0.0
9,languages,NOT NULL,0,0.0


## 4. UNIQUE constraint audit -- `employee_code`

Declared `NOT NULL, UNIQUE`. Not currently in `contracts.py`'s `unique_fields` -- worth checking before deciding whether to add it, same as `branch_code` was.

In [6]:
q("""
    SELECT
        count(*) AS total,
        count(DISTINCT employee_code) AS distinct_employee_code,
        count(*) FILTER (WHERE employee_code IS NULL OR trim(employee_code) = '') AS nulls
    FROM bronze.service_agents
""")

,total,distinct_employee_code,nulls
0,1200,1187,0


In [7]:
q("""
    SELECT employee_code, count(*) AS n, count(DISTINCT agent_id) AS distinct_agent_ids
    FROM bronze.service_agents
    GROUP BY 1 HAVING count(*) > 1
    ORDER BY n DESC
""")

,employee_code,n,distinct_agent_ids
0,E65389,2,2
1,E72566,2,2
2,E15072,2,2
3,E70142,2,2
4,E57622,2,2
5,E28894,2,2
6,E90492,2,2
7,E95303,2,2
8,E53456,2,2
9,E12787,2,2


## 5. Domain checks -- none of these are currently gated in `contracts.py`

All five below have a dictionary-declared value set but zero coverage today. Worth seeing whether they're clean (candidates to add, same as `customer_status`/`product_status`) or already messy (explains why they weren't added, or a new finding).

In [8]:
# native_accent -- dictionary: mexican, colombian, argentine (NOTE: no "neutral" option here,
# unlike customers.detected_accent -- worth seeing if one shows up anyway)
q("SELECT native_accent, count(*) AS n FROM bronze.service_agents GROUP BY 1 ORDER BY n DESC")

,native_accent,n
0,mexican,600
1,colombian,360
2,argentine,240


In [9]:
# agent_type -- dictionary: Phone, In-Person, Digital, Hybrid
q("SELECT agent_type, count(*) AS n FROM bronze.service_agents GROUP BY 1 ORDER BY n DESC")

,agent_type,n
0,Phone,588
1,Digital,251
2,In-Person,230
3,Hybrid,131


In [10]:
# experience_level -- dictionary: Junior, Mid-Senior, Senior, Specialist
q("SELECT experience_level, count(*) AS n FROM bronze.service_agents GROUP BY 1 ORDER BY n DESC")

,experience_level,n
0,Specialist,761
1,Senior,286
2,Mid-Senior,135
3,Junior,18


In [11]:
# agent_status -- dictionary: Active, Vacation, Leave, Inactive
q("SELECT agent_status, count(*) AS n FROM bronze.service_agents GROUP BY 1 ORDER BY n DESC")

,agent_status,n
0,Active,1090
1,Vacation,62
2,Leave,29
3,Inactive,19


In [12]:
# work_shift -- dictionary: Morning, Afternoon, Night, Rotating
q("SELECT work_shift, count(*) AS n FROM bronze.service_agents GROUP BY 1 ORDER BY n DESC")

,work_shift,n
0,Afternoon,418
1,Morning,398
2,Rotating,203
3,Night,181


## 6. `country_of_origin` canonicalization

In [13]:
print("Distinct raw values in Bronze:")
display(q("SELECT country_of_origin, count(*) AS n FROM bronze.service_agents GROUP BY 1 ORDER BY n DESC"))

print("Distinct values in Silver (post-canonicalization):")
display(q("SELECT country_of_origin, count(*) AS n FROM silver.dim_service_agents GROUP BY 1 ORDER BY n DESC"))

country_canonical = {
    "méxico": "México", "mexico": "México", "colombia": "Colombia", "argentina": "Argentina",
    "usa": "USA", "united states": "USA", "spain": "Spain", "brazil": "Brazil", "brasil": "Brazil",
}
raw_values = q("SELECT DISTINCT lower(trim(country_of_origin)) AS v FROM bronze.service_agents WHERE country_of_origin IS NOT NULL")["v"].tolist()
uncovered = [v for v in raw_values if v not in country_canonical]
print("Raw values NOT covered by COUNTRY_CANONICAL:", uncovered or "none -- full coverage")

Distinct raw values in Bronze:


,country_of_origin,n
0,Mexico,600
1,Colombia,360
2,Argentina,240


Distinct values in Silver (post-canonicalization):


,country_of_origin,n
0,México,600
1,Colombia,360
2,Argentina,240


Raw values NOT covered by COUNTRY_CANONICAL: none -- full coverage


## 7. `assigned_branch_id -> branches.branch_id` -- the other half of the branch_id FK investigation

`customers.registration_branch_id` was the first column found orphaned (99.997%). This is the second: `diagnose_branch_fk.ipynb` found `assigned_branch_id` at 69.25% orphaned (831/1,200 populated). Reproducing it directly here, alongside the same "is it just a format/shape mismatch" check used there.

In [14]:
q("""
    SELECT
        count(*) AS total_agents,
        count(*) FILTER (WHERE assigned_branch_id IS NOT NULL) AS branch_id_populated,
        count(*) FILTER (
            WHERE assigned_branch_id IS NOT NULL
            AND NOT EXISTS (SELECT 1 FROM silver.dim_branches b WHERE b.branch_id = a.assigned_branch_id)
        ) AS branch_id_orphans
    FROM silver.dim_service_agents a
""")

,total_agents,branch_id_populated,branch_id_orphans
0,1200,833,831


In [15]:
# Orphaned values' shape -- same format as real branch_id (SUC-XXXXXXXX)? Mostly distinct, or a
# small repeating placeholder set? (Same two checks that ruled out a format bug for customers.)
q("""
    WITH orphans AS (
        SELECT assigned_branch_id FROM silver.dim_service_agents a
        WHERE assigned_branch_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM silver.dim_branches b WHERE b.branch_id = a.assigned_branch_id)
    )
    SELECT
        count(*) AS total_orphans,
        count(DISTINCT assigned_branch_id) AS distinct_orphan_values,
        min(len(assigned_branch_id)) AS min_len, max(len(assigned_branch_id)) AS max_len,
        count(*) FILTER (WHERE assigned_branch_id LIKE 'SUC-%') AS matches_real_format
    FROM orphans
""")

,total_orphans,distinct_orphan_values,min_len,max_len,matches_real_format
0,831,831,12,12,831


## 8. `avg_csat` range sanity -- dictionary: 1-5 scale

In [16]:
q("""
    SELECT
        min(avg_csat) AS min_csat, max(avg_csat) AS max_csat,
        count(*) FILTER (WHERE avg_csat NOT BETWEEN 1 AND 5) AS out_of_declared_range,
        count(*) FILTER (WHERE avg_csat IS NOT NULL) AS populated,
        count(*) AS total
    FROM silver.dim_service_agents
""")

,min_csat,max_csat,out_of_declared_range,populated,total
0,3.5,5.0,0,1066,1200


## 9. `total_monthly_interactions` range sanity

In [17]:
q("""
    SELECT
        min(total_monthly_interactions) AS min_interactions, max(total_monthly_interactions) AS max_interactions,
        count(*) FILTER (WHERE total_monthly_interactions < 0) AS negative_values,
        count(*) FILTER (WHERE total_monthly_interactions IS NOT NULL) AS populated,
        count(*) AS total
    FROM silver.dim_service_agents
""")

,min_interactions,max_interactions,negative_values,populated,total
0,100,800,0,1089,1200


## 10. `hire_date` sanity

In [18]:
q("""
    SELECT
        min(hire_date) AS earliest_hire, max(hire_date) AS latest_hire,
        count(*) FILTER (WHERE hire_date > current_date) AS future_hire_dates,
        count(*) FILTER (WHERE hire_date < DATE '2023-06-17') AS before_dataset_window,
        count(*) FILTER (WHERE hire_date > DATE '2026-06-17') AS after_dataset_window
    FROM silver.dim_service_agents
""")

,earliest_hire,latest_hire,future_hire_dates,before_dataset_window,after_dataset_window
0,2013-06-20,2026-03-17,0,966,0


## 11. `phone`/`email` shape sanity

In [19]:
q("""
    SELECT
        count(*) AS total,
        count(*) FILTER (WHERE phone IS NULL OR trim(phone) = '') AS phone_nulls,
        count(*) FILTER (WHERE phone IS NOT NULL AND trim(phone) <> ''
                          AND regexp_matches(phone, '[0-9]') = false) AS phone_no_digits,
        count(*) FILTER (WHERE email IS NULL OR trim(email) = '') AS email_nulls,
        count(*) FILTER (WHERE email IS NOT NULL AND trim(email) <> '' AND email NOT LIKE '%@%') AS email_missing_at
    FROM bronze.service_agents
""")

,total,phone_nulls,phone_no_digits,email_nulls,email_missing_at
0,1200,69,0,0,0


## 12. `languages` -- comma-separated list, per the spec's docstring note

Kept as a single string in Silver (no split/explode). Checking the actual shape: which combinations appear, and whether the format is consistently comma-separated or has surprises (semicolons, inconsistent spacing, a single-language case, etc.).

In [20]:
q("SELECT languages, count(*) AS n FROM bronze.service_agents GROUP BY 1 ORDER BY n DESC")

,languages,n
0,español,649
1,"español, inglés",422
2,"español, portugués",68
3,"español, inglés, portugués",61


## 13. `specialty` -- free text spot-check

In [21]:
print("specialty distribution:")
display(q("SELECT specialty, count(*) AS n FROM bronze.service_agents GROUP BY 1 ORDER BY n DESC"))
print("null rate:")
display(q("SELECT count(*) AS total, count(*) FILTER (WHERE specialty IS NULL OR trim(specialty) = '') AS nulls FROM bronze.service_agents"))

specialty distribution:


,specialty,n
0,NaN,476
1,Fraudes,105
2,Cobranza,97
3,Soporte Técnico,96
4,Retención,96
5,Créditos,87
6,Inversiones,83
7,Ventas,82
8,Quejas y Reclamos,78


null rate:


,total,nulls
0,1200,476


## 14. Known context + checklist

- **`assigned_branch_id`**: the second of the two known orphaned FK columns (alongside `customers.registration_branch_id`) -- see section 7 for the direct reproduction, and `diagnose_branch_fk.ipynb` for the full original investigation.
- **No domain checks are currently declared for this table in `contracts.py`** -- section 5 covers all five candidates (`native_accent`, `agent_type`, `experience_level`, `agent_status`, `work_shift`); worth deciding after seeing the results which (if any) are worth adding.

## Checklist

- [ ] Column inventory (1): spec/table agree
- [ ] Row reconciliation (2): 1,200 rows as the dictionary declares
- [ ] NOT NULL audit (3): all 13 declared columns populated
- [ ] UNIQUE audit (4): `employee_code` genuinely unique, or note any real collision
- [ ] Domain checks (5): note any value outside each declared set
- [ ] `country_of_origin` (6): full canonicalization coverage
- [ ] FK integrity (7): confirms (or contradicts) the known 69.25% orphan rate; orphan shape matches real format
- [ ] `avg_csat` (8): within 1-5
- [ ] `total_monthly_interactions` (9): no negatives
- [ ] `hire_date` (10): no future dates
- [ ] `phone`/`email` (11): null rates and shape
- [ ] `languages` (12): consistent format
- [ ] `specialty` (13): null rate, no surprises

In [22]:
con.close()
print("closed")

closed
